<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/banners/banner_customer_churn_powerbi_guide.png" width="100%"/>
</div>

## 📖 Préambule

### À qui s'adresse ce guide ?

Ce notebook est ton **support de référence** pour construire le tableau de bord *Customer Churn Analytics — IvoirCom* dans Power BI Desktop. C'est un dashboard de pilotage commercial pour un opérateur télécom mobile en Côte d'Ivoire.

### Comment lire ce guide

| Symbole | Ce qu'il indique |
|---|---|
| 🎯 | Ce que tu sauras faire à la fin de la section |
| 📘 | L'intuition métier ou technique avant de coder |
| 🔧 | Les clics, le code DAX, les paramètres exacts |
| 🎓 | Une méthode opérationnelle pour construire un visuel précis |
| ✅ | Comment vérifier que ton travail est correct |
| ⚠️ | L'erreur courante à éviter |

### Le contexte métier

**IvoirCom** est un opérateur télécom mobile fictif basé à Abidjan, opérant sur 5 villes ivoiriennes (Abidjan, Bouaké, Yamoussoukro, San-Pédro, Korhogo) avec 6 offres au catalogue. La direction commerciale alerte : **12 % de la base abonnés churn chaque trimestre** (25,4 % cumulatif sur 24 mois). Le dashboard pilote 4 enjeux :

1. **Taux de churn** — global et par segment (offre, ville, tranche d'âge), cible interne < 20 %
2. **ARPU** — moyenne par client actif vs churners, pour mesurer la valeur perdue
3. **Réclamations** — signal avant-coureur quand 2+ tickets ne sont pas résolus (93,4 % de churn dans ce segment)
4. **Plan d'action** — segmentation RFM Telecom à 6 segments, top 50 At Risk à recontacter

Le dashboard répond à 5 questions :

| Page | Question |
|---|---|
| 1 — Vue Executive | Quelle est la santé globale de la base abonnés IvoirCom ? |
| 2 — Segments à risque | Qui churne le plus (par offre, ville, âge) ? |
| 3 — Cohortes & Rétention | Quelle génération de souscripteurs tient le mieux ? |
| 4 — Signaux Réclamations | Les plaintes prédisent-elles le départ ? |
| 5 — Plan d'action | Quels clients appeler aujourd'hui (At Risk Top 50) ? |

---
# I — Préparer les fondations

## 1.1 Comprendre les sources de données

### 📘 Concept clé — 5 tables sources + 1 table dérivée RFM

Le projet utilise **5 tables sources** issues du système d'information IvoirCom (CRM + billing + helpdesk) :

- `clients` — 8 000 abonnés propres (après nettoyage 30 doublons + 5 âges négatifs) avec offre, ville, dates de souscription / résiliation
- `offres` — 6 offres au catalogue (Pulse, Connect, Premium, Pro, Étudiant, Senior) avec prix et inclus
- `factures` — 138 284 factures mensuelles sur 24 mois (2 % de montants nuls à exclure de l'ARPU)
- `consommation_mensuelle` — 137 044 lignes voix / SMS / data par client par mois
- `reclamations` — 9 791 tickets support (3 % de délais négatifs, 30 % en statut `ouvert`)

Et **1 table dérivée** produite par le notebook SQL :

- `clients_rfm` — 1 ligne par client avec scores R / F / M (NTILE 1-5). Le `Segment_RFM` final est calculé en **colonne calculée DAX** dans Power BI (cf. section 2.5).

### ⚠️ Piège fréquent — quelle table pour quel KPI ?

- **KPIs volume / churn / âge / ville / offre** ⇒ `clients` (table de référence)
- **ARPU / CA / méthode de paiement** ⇒ `factures`
- **Tickets / délai / type de plainte** ⇒ `reclamations`
- **Voix / SMS / data** ⇒ `consommation_mensuelle`
- **Segment RFM / Top At Risk** ⇒ `clients_rfm`

## 1.2 Importer les CSV

### 📘 Concept clé — pourquoi GitHub raw plutôt que des fichiers locaux ?

Charger depuis une URL `raw.githubusercontent.com` te donne deux superpouvoirs :
1. **Reproductibilité** : tous les apprenants ont exactement la même donnée, à l'octet près.
2. **Mise à jour facile** : si on corrige une coquille dans le CSV, un simple *Actualiser* suffit, aucune ré-installation.

L'inconvénient : il faut une connexion internet au premier chargement. Une fois publié sur le service Power BI, le rapport peut être planifié pour rafraîchir tout seul.

### Les 6 URLs à utiliser

```
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/data/clients.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/data/offres.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/data/factures.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/data/consommation_mensuelle.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/data/reclamations.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/solution/outputs/clients_rfm.csv
```

> 💡 *Si `clients_rfm.csv` n'existe pas encore : exécuter le notebook SQL jusqu'à la section 6.4 (« Export de la table RFM pour Power BI »). La cellule produit le fichier dans `SAVE_PATH` (`./outputs/` en local ou ton Drive sur Colab) via `conn.execute(f"COPY rfm TO '{SAVE_PATH}clients_rfm.csv' (HEADER, DELIMITER ',');")`. Copie-le ensuite dans `dataset/` du projet pour qu'il soit accessible via GitHub raw.*

### 🔧 Procédure pas-à-pas

Pour chacun des 6 CSV : **Accueil → Obtenir les données → Web → Coller l'URL → OK → Charger**.

Dans le panneau **Power Query** :
- Vérifier que les colonnes `date_*` sont bien typées **Date** (pas Texte).
- Sur `factures[montant_fcfa]` et `clients[age]` : type **Nombre entier**.
- Sur `reclamations[delai_resolution_jours]` : type **Nombre entier** (les NULL sont conservés tels quels).
- Renommer la requête `consommation_mensuelle` en `consommation` (plus court).

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/powerbi/tuto/01.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>


## 1.3 Désactiver l'Auto Date/Time

**Fichier → Options → Chargement des données → décocher Date/heure automatique**.

Sans ça, Power BI crée une LocalDateTable cachée pour chaque colonne de date — sur ce projet, c'est 5 tables fantômes en moins (date_facturation, date_souscription, date_resiliation, date_creation, mois).

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/powerbi/tuto/02_options_auto_datetime.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# II — Modéliser les données

## 2.1 Schéma en étoile

### 📘 Concept clé — `clients` est la table pivot, les fact tables sont satellites de Calendrier

```
                  +------------------+
                  |   Calendrier     |   <-- table de dates (DAX)
                  +--------+---------+
                           | 1 (3 actives + 2 inactives vers clients)
                           v
    +--------+ N        N  +------+         N    +------------+
    | offres +-----[1]----+ clients +---[1]----+ clients_rfm |
    +--------+              +-+-+-+              +------------+
                              | | |
                              | | +---[1]--N--> consommation
                              | +---[1]--N--> reclamations
                              +---[1]--N--> factures
```

## 2.2 Créer la table Calendrier

**Modélisation → Nouvelle table** :

```dax
Calendrier = 
ADDCOLUMNS(
    CALENDAR(DATE(2023,1,1), DATE(2025,12,31)),
    "Annee",         YEAR([Date]),
    "Mois_Num",      MONTH([Date]),
    "Mois_Nom",      FORMAT([Date], "mmm", "fr-FR"),
    "Annee_Mois",    FORMAT([Date], "mmm-yy"),
    "Trimestre",     "T" & QUARTER([Date]),
    "Jour_Semaine",  FORMAT([Date], "dddd", "fr-FR")
)
```

## 2.3 Marquer Calendrier comme table de dates

Vue Données → `Calendrier` → **Outils de table → Marquer comme table de dates → colonne Date**.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse/powerbi/tuto/04.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 2.4 Établir les relations

| # | De (1) | Clé | Vers (N) | Clé | État |
|---|---|---|---|---|---|
| 1 | `Calendrier` | `Date` | `factures` | `date_facturation` | Active |
| 2 | `Calendrier` | `Date` | `reclamations` | `date_creation` | Active |
| 3 | `Calendrier` | `Date` | `consommation` | `mois` | Active |
| 4 | `Calendrier` | `Date` | `clients` | `date_souscription` | **Inactive** |
| 5 | `Calendrier` | `Date` | `clients` | `date_resiliation` | **Inactive** |
| 6 | `Calendrier` | `Date` | `clients` | `Cohorte_Mois` | **Inactive** |
| 7 | `offres` | `code_offre` | `clients` | `code_offre` | Active |
| 8 | `clients` | `id_client` | `factures` | `id_client` | Active |
| 9 | `clients` | `id_client` | `reclamations` | `id_client` | Active |
| 10 | `clients` | `id_client` | `consommation` | `id_client` | Active |
| 11 | `clients` | `id_client` | `clients_rfm` | `id_client` | Active (1:1) |

### ⚠️ Pourquoi 3 relations Calendrier ↔ clients en INACTIVE ?

Power BI n'autorise **qu'une seule relation active** entre deux tables, et la rendre active créerait un **chemin ambigu** (factures → clients → Calendrier vs factures → Calendrier direct). On les laisse donc inactives et on les active à la demande dans une mesure via `USERELATIONSHIP()` :

```dax
Souscriptions Mensuelles = 
CALCULATE(
    COUNTROWS(clients),
    USERELATIONSHIP(clients[date_souscription], Calendrier[Date])
)
```

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/powerbi/tuto/02.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 2.5 Colonnes calculées

### Dans `clients` (8 colonnes calculées)

```dax
-- Colonne 1 : flag churner (0 ou 1) — utile pour SUM rapide
Est_Churner = IF(clients[statut] = "resilie", 1, 0)

-- Colonne 2 : tranche d'age metier (5 buckets)
Tranche_Age = 
SWITCH(TRUE(),
    clients[age] >= 18 && clients[age] <= 25, "1. 18-25",
    clients[age] >= 26 && clients[age] <= 35, "2. 26-35",
    clients[age] >= 36 && clients[age] <= 45, "3. 36-45",
    clients[age] >= 46 && clients[age] <= 60, "4. 46-60",
    "5. 60+"
)

-- Colonne 3 : mois de cohorte (1er du mois de souscription)
Cohorte_Mois = DATE(YEAR(clients[date_souscription]), MONTH(clients[date_souscription]), 1)

-- Colonne 4 : pour affichage dans la matrice de cohort
Cohorte_Mois_Label = FORMAT(clients[Cohorte_Mois], "yyyy-MM")

-- Colonne 5 : pour trier Cohorte_Mois_Label chronologiquement (pas alphabetique)
Cohorte_Mois_Sort = YEAR(clients[Cohorte_Mois]) * 100 + MONTH(clients[Cohorte_Mois])

-- Colonne 6 : annee de cohort pour slicer ou filtre visuel
Cohorte_Annee = YEAR(clients[Cohorte_Mois])

-- Colonne 7 : anciennete en mois (date pivot 2025-12-31 si actif)
Anciennete_Mois = 
DATEDIFF(
    clients[date_souscription],
    IF(ISBLANK(clients[date_resiliation]), DATE(2025,12,31), clients[date_resiliation]),
    MONTH
)
```

**🔧 Configuration tri** : Selectionner la colonne `Cohorte_Mois_Label` → **Outils de colonne → Trier par colonne → `Cohorte_Mois_Sort`**. Sinon le tri sera alphabetique (`2024-01, 2024-10, 2024-11, 2024-12, 2024-02, ...`) au lieu de chronologique.

### Dans `clients_rfm` (1 colonne calculée)

```dax
-- Segment RFM derive des 3 scores R / F / M
Segment_RFM = 
SWITCH(TRUE(),
    clients_rfm[R_score] >= 4 && clients_rfm[F_score] >= 4 && clients_rfm[M_score] >= 4, "Champions",
    clients_rfm[R_score] >= 3 && clients_rfm[F_score] >= 3 && clients_rfm[M_score] >= 3, "Loyal",
    clients_rfm[R_score] <= 2 && clients_rfm[M_score] >= 3, "At Risk",
    clients_rfm[R_score] <= 2 && clients_rfm[M_score] <= 2, "Lost",
    clients_rfm[R_score] >= 4 && clients_rfm[M_score] <= 2, "New",
    "Others"
)
```

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/powerbi/tuto/03.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# III — Créer la table `_Mesures`

**Accueil → Entrer des données →** 1 colonne, 1 ligne vide → nommer `_Mesures` → **Charger**.

Après avoir créé ta première mesure et l'avoir glissée dans `_Mesures`, masque la colonne fictive (`Colonne 1` → clic droit → **Masquer dans la vue rapport**).

---
# IV — Construire les ~80 mesures DAX

### Vue d'ensemble des dossiers

| # | Dossier | Mesures | Rôle |
|---|---|---|---|
| 0 | Pilotage Operationnel | 17 | Sous-titres dynamiques + bandeau alerte signal |
| 1 | KPIs Globaux | 9 | Total Clients, Taux Churn %, ARPU global |
| 2 | Evolution | 5 | Souscriptions / résiliations mensuelles avec USERELATIONSHIP |
| 3 | Vue Segmentation | 8 | Taux churn par offre, ville, tranche âge + écart ARPU |
| 4 | Vue Cohortes & Anciennete | 8 | Anciennetés moyennes, médianes, écart, rétention M+N |
| 5 | Vue Reclamations | 11 | Total tickets, % résolus / ouverts / abandonnés, ratio churners/actifs |
| 6 | Vue Plan Action | 19 | RFM segments, Top 50 At Risk, 3 leviers chiffrés |
| _ | _Helpers | 4 | Mesures couleur dynamiques |
| _ | _HTML | 1 | Composition HTML pour visuel HTML Content |

## 4.1 Dossier `0. Pilotage Operationnel` (17 mesures)

```dax
Sous Titre Vue Executive = 
"Etat de la base " & FORMAT([Total Clients], "#,##0") & 
" abonnes  -  Taux churn cumulatif " & FORMAT([Taux Churn %], "0.0") & " %"

Sous Titre Segments = 
"Top offre a risque  -  " & [Nb Offres Au Dessus Seuil] & " offres > 25 % de churn"

Sous Titre Cohortes = 
"Anciennete moyenne actifs " & FORMAT([Anciennete Actifs Mois], "0.0") & 
" mois  vs  churners " & FORMAT([Anciennete Churners Mois], "0.0") & " mois"

Sous Titre Anciennete Actifs = "mediane " & FORMAT([Anciennete Actifs Mediane], "0.0")

Sous Titre Anciennete Churners = "mediane " & FORMAT([Anciennete Churners Mediane], "0.0")

Sous Titre Ecart Anciennete = 
VAR _e = [Ecart Anciennete Actifs vs Churners]
RETURN IF(_e > 0, "duree de vie \u2191 chez actifs",
         IF(_e < 0, "duree de vie \u2193 chez actifs", "duree equivalente"))

Sous Titre Reclamations = 
FORMAT([Total Reclamations], "#,##0") & " tickets  -  " & 
FORMAT([Pct Tickets Non Resolus], "0.0") & " % non resolus (ouvert + abandonne)"

Sous Titre Plan Action = 
"At Risk : " & FORMAT([Nb Clients At Risk], "#,##0") & 
"  -  Top 50 = " & FORMAT([ARPU Top 50 At Risk], "#,##0") & " FCFA / an exposes"

Sous Titre Champions = FORMAT([Pct Champions], "0.0") & " %  \u00B7  ARPU " & FORMAT([ARPU 6m Champions], "#,##0")

Sous Titre At Risk = FORMAT([Pct At Risk], "0.0") & " %  \u00B7  ARPU " & FORMAT([ARPU 6m At Risk], "#,##0")

Sous Titre Lost = FORMAT([Pct Lost], "0.0") & " %  \u00B7  churn " & FORMAT([Taux Churn Lost %], "0.0") & " %"

Sous Titre Top 50 At Risk = 
"Tous abonnes Pro  \u00B7  ARPU mensuel " & FORMAT([ARPU Min Top 50 At Risk], "#,##0") & 
" a " & FORMAT([ARPU Max Top 50 At Risk], "#,##0") & " FCFA"

Caption ARPU Annuel Defendre = "FCFA / an si rien n'est fait"

Titre Signal Phare = "Signal phare - churn parmi 2+ tickets non resolus"

Sous Titre Segment Risque = 
"Segment a risque (" & FORMAT([Nb Clients 2 Plus Tickets Non Resolus], "#,##0") & " clients)"

Sous Titre Base Globale = "Base globale (" & FORMAT([Total Clients], "#,##0") & ")"

Bandeau Alerte = 
VAR _n = [Nb Clients 2 Plus Tickets Non Resolus]
VAR _taux = [Taux Churn Segment Risque %]
RETURN IF(
    _n > 0,
    "\u26A0 " & FORMAT(_n, "#,##0") & " clients a 2+ tickets non resolus  -  Taux observe " & 
        FORMAT(_taux, "0.0") & " %  -  Action immediate requise",
    "\u2713 Aucun signal critique en cours - Niveau de service nominal"
)

Total 3 Leviers Range M FCFA = 
VAR _total = [Total 3 Leviers M FCFA]
VAR _min = ROUND(_total * 0.95, 0)
VAR _max = ROUND(_total * 1.05, 0)
RETURN "\u2248 " & _min & "-" & _max & " M FCFA"
```

### 📘 Pourquoi des sous-titres dynamiques ?

Au lieu d'écrire « Vue Executive » en dur en haut de page, on affiche un **sous-titre qui se met à jour avec les filtres**. Quand l'utilisateur active le slicer Année 2025, le sous-titre recalcule automatiquement le taux de churn 2025 — c'est ce qui transforme un dashboard statique en outil d'investigation.

## 4.2 Dossier `1. KPIs Globaux` (9 mesures)

```dax
Total Clients = COUNTROWS(clients)

Total Actifs = CALCULATE([Total Clients], clients[statut] = "actif")

Total Resilies = CALCULATE([Total Clients], clients[statut] = "resilie")

Taux Churn % = DIVIDE([Total Resilies], [Total Clients]) * 100

Taux Churn Ratio = DIVIDE([Total Resilies], [Total Clients])

Taux Conforme Ratio = 1 - [Taux Churn Ratio]

ARPU Mensuel Moyen = 
AVERAGEX(
    SUMMARIZE(
        FILTER(factures, factures[montant_fcfa] > 0),
        factures[id_client]
    ),
    CALCULATE(AVERAGE(factures[montant_fcfa]))
)

ARPU Actifs = CALCULATE([ARPU Mensuel Moyen], clients[statut] = "actif")

ARPU Churners = CALCULATE([ARPU Mensuel Moyen], clients[statut] = "resilie")
```

### ⚠️ Pourquoi `Taux Churn %` ET `Taux Churn Ratio` ?

- `Taux Churn %` retourne **25,4** (déjà × 100, format `0.0\"%\"`) — pour cards et bandeau
- `Taux Churn Ratio` retourne **0,254** (ratio 0-1, format Pourcentage) — pour les barres empilées 100 %

Cette dualité existe car certains visuels Power BI attendent un ratio (les barres empilées 100 %), d'autres préfèrent un nombre déjà formaté.

## 4.3 Dossier `2. Evolution` (5 mesures)

```dax
Souscriptions Mensuelles = 
CALCULATE(
    COUNTROWS(clients),
    USERELATIONSHIP(clients[date_souscription], Calendrier[Date])
)

Resiliations Mensuelles = 
CALCULATE(
    COUNTROWS(clients),
    USERELATIONSHIP(clients[date_resiliation], Calendrier[Date]),
    NOT(ISBLANK(clients[date_resiliation]))
)

Churn Net Mois = [Resiliations Mensuelles] - [Souscriptions Mensuelles]

Souscriptions YTD = TOTALYTD([Souscriptions Mensuelles], Calendrier[Date])

Resiliations YTD = TOTALYTD([Resiliations Mensuelles], Calendrier[Date])
```

### 📘 USERELATIONSHIP — pourquoi 2 relations inactives ?

Le même `Calendrier[Date]` doit pouvoir filtrer **soit** les souscriptions, **soit** les résiliations selon le contexte. Power BI n'autorise qu'**une relation active** entre deux tables : on garde donc les 2 relations inactives et on choisit dynamiquement avec `USERELATIONSHIP()` à l'intérieur de chaque mesure.

## 4.4 Dossier `3. Vue Segmentation` (8 mesures)

```dax
Nb Clients Offre = COUNTROWS(clients)

Taux Churn Offre % = 
DIVIDE(
    CALCULATE(COUNTROWS(clients), clients[statut] = "resilie"),
    COUNTROWS(clients)
) * 100

Rang Offre Churn = 
RANKX(ALL(offres[code_offre]), CALCULATE([Taux Churn Offre %]), , DESC, DENSE)

Nb Offres Au Dessus Seuil = 
VAR _seuil = 25
RETURN
COUNTROWS(
    FILTER(
        VALUES(offres[code_offre]),
        CALCULATE([Taux Churn Offre %]) > _seuil
    )
)

Taux Churn Ville % = 
DIVIDE(
    CALCULATE(COUNTROWS(clients), clients[statut] = "resilie"),
    COUNTROWS(clients)
) * 100

Taux Churn Tranche Age % = 
DIVIDE(
    CALCULATE(COUNTROWS(clients), clients[statut] = "resilie"),
    COUNTROWS(clients)
) * 100

Top Ville Churn = 
VAR _t = 
    TOPN(1,
        ADDCOLUMNS(VALUES(clients[ville]), "@taux", CALCULATE([Taux Churn %])),
        [@taux], DESC
    )
RETURN MAXX(_t, clients[ville]) & " (" & FORMAT(MAXX(_t, [@taux]), "0.0") & " %)"

Ecart ARPU Actifs vs Churners % = 
DIVIDE([ARPU Actifs], [ARPU Churners]) - 1
```

### ⚠️ Piège — `RANKX` qui renvoie 1 partout

Sur un visuel table, sans le `CALCULATE([Taux Churn Offre %])` interne, `RANKX` n'arrive pas à transitionner du contexte de ligne au contexte de filtre. Tous les rangs renvoient 1. **Toujours wrapper la mesure dans CALCULATE quand RANKX itère sur ALL.**

## 4.5 Dossier `4. Vue Cohortes & Anciennete` (8 mesures)

```dax
Anciennete Mois Moyenne = AVERAGE(clients[Anciennete_Mois])

Anciennete Actifs Mois = 
CALCULATE([Anciennete Mois Moyenne], clients[statut] = "actif")

Anciennete Churners Mois = 
CALCULATE([Anciennete Mois Moyenne], clients[statut] = "resilie")

Anciennete Actifs Mediane = 
CALCULATE(MEDIAN(clients[Anciennete_Mois]), clients[statut] = "actif")

Anciennete Churners Mediane = 
CALCULATE(MEDIAN(clients[Anciennete_Mois]), clients[statut] = "resilie")

Ecart Anciennete Actifs vs Churners = 
[Anciennete Actifs Mois] - [Anciennete Churners Mois]

Retention Cohorte Pct = 
VAR _M = MAX(clients[Anciennete_Mois])
RETURN
DIVIDE(
    CALCULATE(
        COUNTROWS(clients),
        FILTER(
            ALLEXCEPT(clients,
                clients[Cohorte_Mois], clients[Cohorte_Mois_Label],
                clients[Cohorte_Mois_Sort], clients[Cohorte_Annee]
            ),
            clients[Anciennete_Mois] >= _M
        )
    ),
    CALCULATE(
        COUNTROWS(clients),
        ALLEXCEPT(clients,
            clients[Cohorte_Mois], clients[Cohorte_Mois_Label],
            clients[Cohorte_Mois_Sort], clients[Cohorte_Annee]
        )
    )
) * 100

Pct Cohortes Saines = 
VAR _toutes_cohortes = 
    FILTER(SUMMARIZE(clients, clients[Cohorte_Mois]), YEAR(clients[Cohorte_Mois]) = 2024)
VAR _saines = FILTER(_toutes_cohortes, [Retention Cohorte Pct] >= 80)
RETURN DIVIDE(COUNTROWS(_saines), COUNTROWS(_toutes_cohortes)) * 100
```

**Format string** : ajouter `0.0" mois"` sur les 5 mesures d'ancienneté pour afficher `25,9 mois`. Format `+0.0" mois";-0.0" mois";0" mois"` sur l'écart pour le signe explicite (+8,7 mois).

### 📘 Construction de la heatmap rétention

On utilisera un visuel **Matrix** (page 3) avec :
- **Lignes** : `clients[Cohorte_Mois_Label]` (formaté yyyy-MM, trié par `Cohorte_Mois_Sort`)
- **Colonnes** : `clients[Anciennete_Mois]` (filtré 0 à 11)
- **Valeurs** : `[Retention Cohorte Pct]`
- **Filtre visuel** : `clients[Cohorte_Annee] = 2024`
- **Mise en forme conditionnelle** : dégradé rouge (50%) → vert (100%)

### 🎓 MÉTHODE — Pourquoi `MAX(Anciennete_Mois)` dans la mesure ?

Sans `VAR _M = MAX(clients[Anciennete_Mois])`, la mesure ne tient pas compte du contexte colonne et retourne **74,6 % partout** (le ratio actifs/total global). Le `MAX` capture la valeur du contexte avant que `ALLEXCEPT` ne supprime tous les filtres sauf ceux de Cohorte. Le filtre `Anciennete_Mois >= _M` re-applique alors la logique de survivance.

## 4.6 Dossier `5. Vue Reclamations` (11 mesures)

```dax
Total Reclamations = COUNTROWS(reclamations)

Nb Reclamations Resolues = 
CALCULATE(COUNTROWS(reclamations), reclamations[statut] = "resolu")

Nb Reclamations Ouvertes = 
CALCULATE(COUNTROWS(reclamations), reclamations[statut] = "ouvert")

Nb Reclamations Abandonnees = 
CALCULATE(COUNTROWS(reclamations), reclamations[statut] = "abandonne")

Pct Tickets Resolus = 
DIVIDE([Nb Reclamations Resolues], [Total Reclamations]) * 100

Pct Tickets Non Resolus = 
DIVIDE([Nb Reclamations Ouvertes] + [Nb Reclamations Abandonnees], [Total Reclamations]) * 100

Delai Resolution Moyen Jours = 
CALCULATE(AVERAGE(reclamations[delai_resolution_jours]), reclamations[statut] = "resolu")

Tickets Moyens Par Client = 
DIVIDE([Total Reclamations], DISTINCTCOUNT(reclamations[id_client]))

Tickets Moyens Churners = 
VAR _t = 
    SUMMARIZE(
        FILTER(clients, clients[statut] = "resilie"),
        clients[id_client],
        "@n", CALCULATE(COUNTROWS(reclamations))
    )
RETURN AVERAGEX(_t, [@n] + 0)

Tickets Moyens Actifs = 
VAR _t = 
    SUMMARIZE(
        FILTER(clients, clients[statut] = "actif"),
        clients[id_client],
        "@n", CALCULATE(COUNTROWS(reclamations))
    )
RETURN AVERAGEX(_t, [@n] + 0)

Ratio Tickets Churners vs Actifs = 
DIVIDE([Tickets Moyens Churners], [Tickets Moyens Actifs])
```

### 🎓 MÉTHODE — Pourquoi `+ 0` après `[@n]` ?

Pour les clients qui n'ont **aucun ticket**, `CALCULATE(COUNTROWS(reclamations))` retourne `BLANK`. Sans le `+ 0`, `AVERAGEX` ignore ces lignes et la moyenne est biaisée vers le haut. Le `+ 0` force la conversion `BLANK → 0` et la moyenne devient correcte.

## 4.7 Dossier `6. Vue Plan Action` (19 mesures)

### Signal 2+ tickets non résolus (3 mesures)

```dax
Nb Clients 2 Plus Tickets Non Resolus = 
VAR _segments = 
    FILTER(
        SUMMARIZE(
            FILTER(reclamations, reclamations[statut] IN {"ouvert", "abandonne"}),
            reclamations[id_client],
            "@n", COUNT(reclamations[id_ticket])
        ),
        [@n] >= 2
    )
RETURN COUNTROWS(_segments)

Taux Churn Segment Risque % = 
VAR _ids_a_risque = 
    SELECTCOLUMNS(
        FILTER(
            SUMMARIZE(
                FILTER(reclamations, reclamations[statut] IN {"ouvert", "abandonne"}),
                reclamations[id_client],
                "@n", COUNT(reclamations[id_ticket])
            ),
            [@n] >= 2
        ),
        "id_client", reclamations[id_client]
    )
VAR _total_segment = COUNTROWS(_ids_a_risque)
VAR _churners_segment = 
    CALCULATE(
        COUNTROWS(clients),
        clients[statut] = "resilie",
        TREATAS(_ids_a_risque, clients[id_client])
    )
RETURN DIVIDE(_churners_segment, _total_segment) * 100

Ratio Signal vs Base = 
DIVIDE([Taux Churn Segment Risque %], [Taux Churn %])
```

### Segments RFM — comptes & pourcentages (8 mesures)

```dax
Nb Clients Champions = CALCULATE(COUNTROWS(clients_rfm), clients_rfm[Segment_RFM] = "Champions")
Nb Clients Loyal     = CALCULATE(COUNTROWS(clients_rfm), clients_rfm[Segment_RFM] = "Loyal")
Nb Clients At Risk   = CALCULATE(COUNTROWS(clients_rfm), clients_rfm[Segment_RFM] = "At Risk")
Nb Clients Lost      = CALCULATE(COUNTROWS(clients_rfm), clients_rfm[Segment_RFM] = "Lost")
Nb Clients New       = CALCULATE(COUNTROWS(clients_rfm), clients_rfm[Segment_RFM] = "New")

Pct Champions = DIVIDE([Nb Clients Champions], COUNTROWS(clients_rfm)) * 100
Pct At Risk   = DIVIDE([Nb Clients At Risk],   COUNTROWS(clients_rfm)) * 100
Pct Lost      = DIVIDE([Nb Clients Lost],      COUNTROWS(clients_rfm)) * 100
```

### ARPU & churn par segment (4 mesures)

```dax
ARPU 6m Champions = 
CALCULATE(AVERAGE(clients_rfm[arpu_6m]), clients_rfm[Segment_RFM] = "Champions")

ARPU 6m At Risk = 
CALCULATE(AVERAGE(clients_rfm[arpu_6m]), clients_rfm[Segment_RFM] = "At Risk")

Taux Churn Lost % = 
VAR _ids_lost = 
    SELECTCOLUMNS(
        FILTER(clients_rfm, clients_rfm[Segment_RFM] = "Lost"),
        "id_client", clients_rfm[id_client]
    )
VAR _total = COUNTROWS(_ids_lost)
VAR _churners = 
    CALCULATE(
        COUNTROWS(clients),
        clients[statut] = "resilie",
        TREATAS(_ids_lost, clients[id_client])
    )
RETURN DIVIDE(_churners, _total) * 100

ARPU Top 50 At Risk = 
VAR _top50 = 
    TOPN(
        50,
        FILTER(clients_rfm, clients_rfm[Segment_RFM] = "At Risk"),
        clients_rfm[arpu_6m], DESC
    )
RETURN SUMX(_top50, clients_rfm[arpu_6m]) * 12
```

### Top 50 At Risk — détails (3 mesures)

```dax
ARPU Top 50 At Risk M FCFA = [ARPU Top 50 At Risk] / 1000000

ARPU Min Top 50 At Risk = 
VAR _top50 = TOPN(50, FILTER(clients_rfm, clients_rfm[Segment_RFM] = "At Risk"), clients_rfm[arpu_6m], DESC)
RETURN MINX(_top50, clients_rfm[arpu_6m])

ARPU Max Top 50 At Risk = 
VAR _top50 = TOPN(50, FILTER(clients_rfm, clients_rfm[Segment_RFM] = "At Risk"), clients_rfm[arpu_6m], DESC)
RETURN MAXX(_top50, clients_rfm[arpu_6m])
```

### 3 leviers chiffrés (4 mesures)

```dax
Levier 1 Retention Tickets M FCFA = 
VAR _clients_risque = [Nb Clients 2 Plus Tickets Non Resolus]
VAR _taux_recup = 0.30
VAR _arpu_mois = [ARPU Mensuel Moyen]
VAR _mois = 12
RETURN _clients_risque * _taux_recup * _arpu_mois * _mois / 1000000

Levier 2 Migration Pulse Connect M FCFA = 
VAR _clients_cibles = 800
VAR _upsell_mois = 4500
VAR _mois = 12
RETURN _clients_cibles * _upsell_mois * _mois / 1000000

Levier 3 Programme At Risk M FCFA = [ARPU Top 50 At Risk] * 0.50 / 1000000

Total 3 Leviers M FCFA = 
[Levier 1 Retention Tickets M FCFA] + 
[Levier 2 Migration Pulse Connect M FCFA] + 
[Levier 3 Programme At Risk M FCFA]
```

### 📘 TREATAS pour le signal 2+ tickets non résolus

`TREATAS` permet d'**injecter une liste virtuelle de `id_client`** (calculée depuis reclamations) comme filtre sur la table `clients`. C'est le seul moyen propre de croiser un agrégat de reclamations avec un comptage de clients sans créer de relation supplémentaire.

## 4.8 Dossier `_Helpers` — couleurs dynamiques (4 mesures)

```dax
Color Statut = 
SWITCH(SELECTEDVALUE(clients[statut]),
    "actif",   "#10D9A3",
    "resilie", "#FF4D6D",
    "#888780"
)

Color Offre Churn = 
VAR _t = [Taux Churn Offre %]
RETURN
SWITCH(TRUE(),
    _t >= 30, "#FF4D6D",   -- rouge danger
    _t >= 25, "#FFB547",   -- orange warning
    "#10D9A3"              -- vert OK
)

Color Segment RFM = 
SWITCH(SELECTEDVALUE(clients_rfm[Segment_RFM]),
    "Champions", "#10D9A3",
    "Loyal",     "#534AB7",
    "At Risk",   "#FFB547",
    "Lost",      "#FF4D6D",
    "New",       "#7891B5",
    "#888780"
)

Color Bandeau Alerte = 
IF([Nb Clients 2 Plus Tickets Non Resolus] > 0, "#3D1421", "#0F2C20")
```

### 🎓 MÉTHODE — appliquer une couleur dynamique

Sur un visuel (carte, barre, table) :
1. Format → **Couleur de remplissage** (ou Couleur du texte) → **fx**
2. **Mettre en forme par : Valeur du champ**
3. Choisir la mesure couleur (`[Color Offre Churn]`, etc.)

La cellule prend la couleur retournée par la mesure selon le contexte de ligne.

## 4.9 Dossier `_HTML` — visuel HTML Content (1 mesure)

Cette mesure compose **directement la card des 3 leviers chiffrés** en HTML, à utiliser dans le visuel **HTML Content** (Daniel Marsh-Patrick depuis AppSource). Elle remplace les 3 lignes de carte multi-lignes habituelles par un seul visuel pixel-perfect.

```dax
HTML Plan Action = 
VAR _l1 = FORMAT([Levier 1 Retention Tickets M FCFA], "0")
VAR _l2 = FORMAT([Levier 2 Migration Pulse Connect M FCFA], "0")
VAR _l3 = FORMAT([Levier 3 Programme At Risk M FCFA], "0.0")
VAR _n_tickets = FORMAT([Nb Clients 2 Plus Tickets Non Resolus], "#,##0")
VAR _t = [Total 3 Leviers M FCFA]
VAR _total_min = FORMAT(ROUND(_t * 0.95, 0), "0")
VAR _total_max = FORMAT(ROUND(_t * 1.05, 0), "0")
RETURN
"<div style=""color:#fff;font-family:Segoe UI,sans-serif;""><div style=""background:#1A2436;border:1px solid #2A3548;border-radius:8px;padding:12px 14px;"">[... voir cellule de la mesure dans Power BI Desktop ...]"
```

### 🎓 MÉTHODE — Installer et utiliser HTML Content

1. **Power BI Desktop** → menu Visualisations → **Obtenir d'autres visuels** → AppSource
2. Chercher **HTML Content** (par Daniel Marsh-Patrick) → Ajouter
3. Insérer le visuel sur Page 5 (largeur pleine, hauteur ~280 px)
4. Glisser `[HTML Plan Action]` dans **Values**
5. Le HTML est rendu dynamiquement avec les vraies valeurs

### ⚠️ Règles HTML Content

- Toute la mesure doit être sur **une seule ligne RETURN** (pas de saut de ligne dans la chaîne)
- Les guillemets doubles dans le HTML sont escapés en `""` (DAX double-quote)
- Pas d'apostrophe dans les valeurs CSS (utiliser `font-family:Segoe UI` sans quotes)
- Pour le caractère ≈ : `\u2248` ou simplement `~`

---
# V — Design system

## 5.1 Charte Customer Churn — Dark + Orange chaleureux

Le dashboard utilise un **fond sombre navy** avec accents **orange chaleureux** (charte IvoirCom).

| Rôle | Hex | Usage |
|---|---|---|
| **Fond page** | `#0B1421` | Canvas Power BI principal |
| Card background | `#1A2436` | Background des visuels et cartes KPI |
| Card elevated | `#232F45` | Pastilles d'icônes, zones surbrillance |
| **Primaire (orange)** | `#E94E1B` | Logo, navbar active, hero numbers, accents |
| Sidebar foncé | `#A8350E` | Footer, navbar foncée |
| Vert OK / Champions | `#10D9A3` | Conforme, Champions, succès |
| Orange warning | `#FFB547` | At Risk, alertes intermédiaires |
| Rouge danger | `#FF4D6D` | Churn, Lost, alertes critiques |
| Violet RFM Loyal | `#534AB7` | Segment Loyal |
| Bleu RFM New | `#7891B5` | Segment New |
| Texte principal | `#FFFFFF` | Hero numbers, titres |
| Texte secondaire | `#B0B8C7` | Labels, sous-titres |
| Texte tertiaire | `#6B7B95` | Footer, axes, légendes |
| Bordure subtile | `#2A3548` | Séparation cards et zones |
| Bandeau alerte (fond) | `#3D1421` | Zone signal critique |

### Typographie

| Élément | Police | Taille | Poids |
|---|---|---|---|
| Titre de page | Georgia | 36 | 700 |
| Sous-titre | Segoe UI | 13 | 300 |
| Hero KPI | Segoe UI | 32 | 700 |
| Label KPI | Segoe UI | 12 | 400 (gris clair `#B0B8C7`) |
| Navbar item | Segoe UI | 13 | 500 |

### 🔧 Application sur la page Power BI

1. Sélectionner la page → **Format de la page → Canevas**
2. **Couleur d'arrière-plan** → Hex `#0B1421` → Transparence 0 %
3. Si tu utilises un PNG d'arrière-plan, mets la couleur du canevas **dessous** pour éviter tout flash blanc au chargement

## 5.2 Mockup PowerPoint → fonds PNG d'arrière-plan

### 📘 Concept clé

Power BI gère mal les arrière-plans complexes. Méthode pro : dessiner dans **PowerPoint** (mockup vierge), exporter en **PNG haute résolution** (1280×720), importer comme **arrière-plan de page**, poser les visuels Power BI **par-dessus**.

### Ce que le mockup PPTX doit contenir

✅ Logo IvoirCom (carré orange `#E94E1B` + icône smartphone blanc) en haut-gauche, navbar horizontale en haut avec icônes (target / bar / network / ticket / bolt), footer DataProjectLab orange foncé, cards `#1A2436` avec bordure subtile `#2A3548`.

❌ Pas de titre de page, pas de slicers, pas de KPI valeurs, pas de chart data.

### 🔧 Méthode 1 — Export PNG depuis PowerPoint à 150 DPI

1. **Win + R** → `regedit` → `HKEY_CURRENT_USER\Software\Microsoft\Office\16.0\PowerPoint\Options`
2. Clic droit → **Nouveau** → **Valeur DWORD (32 bits)** → Nom : `ExportBitmapResolution`, Valeur : `150`
3. Redémarrer PowerPoint, **Fichier → Enregistrer sous → PNG → Toutes les diapositives**

### 🔧 Méthode 2 — CloudConvert

[cloudconvert.com/pptx-to-png](https://cloudconvert.com/pptx-to-png) → upload `mockup_customer_churn_dark.pptx` → 150 DPI → 1280×720.

### Renommage final

```
bg-01-vue-executive.png
bg-02-segments.png
bg-03-cohortes.png
bg-04-reclamations.png
bg-05-plan-action.png
```

### 🔧 Application dans Power BI

1. Sélectionner la page → **Format de la page**
2. **Arrière-plan de la page** → **Ajouter une image** → choisir le PNG
3. **Ajustement** → **Adapter** · **Transparence** → **0 %**


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/powerbi/tuto/04.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# VI — Construire les 5 pages

Cette partie détaille **chaque visuel** avec sa configuration exacte (type, axes, couleurs, étiquettes) et les **méthodes Power BI** non-triviales nécessaires pour le rendu final.

## 6.1 Page 1 — Vue Executive

> *« Quelle est la santé globale de la base abonnés IvoirCom ? »*

**Sous-titre dynamique** (haut de page) : `[Sous Titre Vue Executive]` → `Etat de la base 8 000 abonnes - Taux churn cumulatif 25,4 %`

**1. Bandeau alerte signal**
- Type : Carte avec valeur dynamique
- Texte : `[Bandeau Alerte]`
- Fond : `[Color Bandeau Alerte]` via fx → Valeur du champ
- Bordure gauche : 4px rouge `#FF4D6D`

**2-5. 4 KPI cards**
- KPI 1 — **Total Clients** : icône 👥, valeur `[Total Clients]` (8 000) blanc 32pt
- KPI 2 — **Taux Churn** : icône ⚠️, valeur `[Taux Churn %]` (25,4 %), couleur `#FF4D6D` 32pt
- KPI 3 — **ARPU Actifs** : icône 💰, valeur `[ARPU Actifs]` (7 959 FCFA), couleur `#10D9A3` 32pt
- KPI 4 — **Total Réclamations** : icône 📋, valeur `[Total Reclamations]` (9 791), couleur `#FFB547` 32pt

**6. Donut — Répartition Actifs / Résiliés**
- Type : Anneau
- Catégorie : `clients[statut]`
- Valeur : `[Total Clients]`
- Couleurs : actif vert `#10D9A3`, resilie rouge `#FF4D6D`
- Trou central : 65 %

**7. Évolution mensuelle Souscriptions vs Résiliations**
- Type : **Graphique en courbes** (2 séries)
- Axe X : `Calendrier[Annee_Mois]`
- Valeurs : `[Souscriptions Mensuelles]` (vert) et `[Resiliations Mensuelles]` (rouge)

**8. Taux de churn par offre**
- Type : Barres verticales
- Axe Y : `[Taux Churn Offre %]`
- Axe X : `offres[libelle]`
- Couleur barres : `[Color Offre Churn]` via fx → Valeur du champ
- Étiquettes : valeur en `%` à droite des barres
- Tri : descendant

> 🎓 **MÉTHODE — KPI card avec icône**
>
> Power BI ne permet pas d'icône native dans une carte. Astuce :
>
> 1. Insérer une **Forme** (carré arrondi) avec la couleur de la pastille (ex `#232F45`).
> 2. Insérer un **Texte** avec l'emoji ou un caractère Unicode glyph (📋, ⚠️, 💰).
> 3. Empiler la carte de valeur par-dessus.


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/powerbi/tuto/01-page-exec.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.2 Page 2 — Segments à risque

> *« Qui churne le plus (par offre, ville, âge) ? »*

**Sous-titre dynamique** : `[Sous Titre Segments]`

**1-3. 3 KPI cards**
- KPI 1 — **Offres au-dessus seuil 25 %** : `[Nb Offres Au Dessus Seuil]` (2) rouge 32pt
- KPI 2 — **Top ville à risque** : `[Top Ville Churn]` (« Korhogo (27,8 %) ») 24pt
- KPI 3 — **Écart ARPU actifs / churners** : `[Ecart ARPU Actifs vs Churners %]` (`+71,7 %`) 32pt orange. Format string : `+0.0%;-0.0%`

**4. Taux churn par offre (barres horizontales)**
- Identique au visuel 8 de la page 1, mais en plus grand
- Ligne pointillée à 25 % (seuil critique) en `#FFB547`

**5. Taux churn par ville (barres horizontales)**
- Type : Barres horizontales
- Axe Y : `clients[ville]`
- Axe X : `[Taux Churn Ville %]`
- Couleur : `#534AB7` uniforme

**6. Heatmap croisée Offre × Ville**
- Type : **Matrix**
- Lignes : `clients[ville]`
- Colonnes : `offres[code_offre]`
- Valeurs : `[Taux Churn Offre %]`
- Mise en forme conditionnelle : **Mettre en échelle des couleurs** (vert `#10D9A3` à 0, orange `#FFB547` à 25, rouge `#FF4D6D` à 40)

**7. Taux churn par tranche d'âge (barres verticales)**
- Axe X : `clients[Tranche_Age]`
- Axe Y : `[Taux Churn Tranche Age %]`
- Couleur : orange `#E94E1B`

> 🎓 **MÉTHODE — Heatmap dans une matrice Power BI**
>
> 1. Visuel **Matrix**
> 2. Format → **Éléments de cellule** → Activer **Couleur d'arrière-plan**
> 3. fx → **Mise en échelle des couleurs** → Valeurs : Min `#10D9A3`, Centre `#FFB547` (à 25), Max `#FF4D6D`
> 4. Format → **Total général** : Désactiver pour ne pas polluer la heatmap


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/powerbi/tuto/02-page-segment.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.3 Page 3 — Cohortes & Rétention

> *« Quelle génération de souscripteurs tient le mieux ? »*

**Sous-titre dynamique** : `[Sous Titre Cohortes]`

**1-3. 3 KPI cards**
- KPI 1 — **Ancienneté actifs** : `[Anciennete Actifs Mois]` (`25,9 mois`) vert 32pt + sous-texte `[Sous Titre Anciennete Actifs]` (`mediane 26,2`)
- KPI 2 — **Ancienneté churners** : `[Anciennete Churners Mois]` (`17,2 mois`) rouge 32pt + sous-texte `[Sous Titre Anciennete Churners]` (`mediane 15,1`)
- KPI 3 — **Écart actifs / churners** : `[Ecart Anciennete Actifs vs Churners]` (`+8,7 mois`) orange 32pt + sous-texte `[Sous Titre Ecart Anciennete]` (`duree de vie ↑ chez actifs`)

**Slicer dédié** : `clients[Cohorte_Annee]` (vignette `2024 / 2025`) — filtre uniquement la heatmap, pas le reste de la page.

**4. Heatmap rétention 12 cohortes × 12 mois**
- Type : **Matrix**
- Lignes : `clients[Cohorte_Mois_Label]` (trié par `Cohorte_Mois_Sort`)
- Colonnes : `clients[Anciennete_Mois]`
- Valeurs : `[Retention Cohorte Pct]`
- Filtre visuel : `clients[Cohorte_Annee] = 2024` ET `clients[Anciennete_Mois] <= 11`
- Format conditionnel : Min `#FF4D6D` à 50 %, Centre `#FFB547` à 80 %, Max `#10D9A3` à 100 %
- Format → Total général : Désactivé

**5. Courbe rétention médiane**
- Type : Graphique en courbes
- Axe X : `clients[Anciennete_Mois]` (0 à 11)
- Axe Y : `[Retention Cohorte Pct]`
- Couleur ligne : orange `#E94E1B` épaisseur 2px

**6. Table — Top cohortes les plus solides / fragiles**
- Colonnes : `Cohorte_Mois_Label`, taille initiale, rétention M+11, écart vs médiane
- Tri : descendant sur rétention M+11

> 🎓 **MÉTHODE — Pourquoi le slicer Cohorte_Annee plutôt que Calendrier[Annee] ?**
>
> La relation `clients[Cohorte_Mois] ↔ Calendrier[Date]` est **inactive** (chemin ambigu via factures). Le slicer `Calendrier[Annee]` ne peut donc pas filtrer la heatmap. Solution : utiliser la colonne calculée `clients[Cohorte_Annee]` (extraite de Cohorte_Mois) qui filtre directement `clients` sans passer par Calendrier.


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/powerbi/tuto/03-cohorts.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.4 Page 4 — Signaux Réclamations

> *« Les plaintes prédisent-elles le départ ? »*

**Sous-titre dynamique** : `[Sous Titre Reclamations]`

**1-3. 3 KPI cards**
- KPI 1 — **Total Réclamations** : `[Total Reclamations]` (9 791) orange 32pt
- KPI 2 — **% Tickets Non Résolus** : `[Pct Tickets Non Resolus]` (47,8 %) rouge 32pt
- KPI 3 — **Délai Résolution Moyen** : `[Delai Resolution Moyen Jours]` (~4 j) bleu 32pt

**4. Barres empilées Type × Statut**
- Type : Histogramme empilé
- Axe X : `reclamations[type]`
- Légende : `reclamations[statut]`
- Valeur : `COUNTROWS(reclamations)`
- Couleurs : resolu vert `#10D9A3`, ouvert orange `#FFB547`, abandonne rouge `#FF4D6D`

**5. Comparaison tickets moyens Actifs vs Churners**
- Type : Barres horizontales
- 2 mesures : `[Tickets Moyens Actifs]` (~0,78) et `[Tickets Moyens Churners]` (~2,53)
- Couleurs respectives vert et rouge
- Sous-texte : Ratio churners/actifs `[Ratio Tickets Churners vs Actifs]` (`×3,2`)

**6. Bandeau hero — Signal phare**

Composé de 4 mesures :
- **Titre** : `[Titre Signal Phare]` (`Signal phare - churn parmi 2+ tickets non resolus`)
- **Sous-titre gauche** : `[Sous Titre Segment Risque]` (`Segment a risque (1 195 clients)`)
- **Valeur gauche** : `[Taux Churn Segment Risque %]` (93,4 %) — 32pt rouge
- **Sous-titre droit** : `[Sous Titre Base Globale]` (`Base globale (8 000)`)
- **Valeur droite** : `[Taux Churn %]` (25,4 %) — 32pt blanc
- **Écart** : `[Ratio Signal vs Base]` (`×3,7`) — 24pt orange

Fond : `#3D1421` (rouge bordeaux foncé), bordure 2px `#FF4D6D`, coins 4px.

**7. Top 10 clients à plus de tickets non résolus**
- Type : Table
- Colonnes : `id_client`, `code_offre`, `ville`, nb tickets ouverts, nb tickets abandonnés, ARPU 6m
- Tri : descendant nb tickets
- Couleur conditionnelle : ARPU > 10 000 FCFA en rouge


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/powerbi/tuto/04-reclamation.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.5 Page 5 — Plan d'Action

> *« Quels clients appeler aujourd'hui ? »*

**Sous-titre dynamique** : `[Sous Titre Plan Action]`

**1-3. 3 KPI cards RFM** (avec bordure gauche colorée 2px)
- KPI 1 — **Champions** (bordure verte) : `[Nb Clients Champions]` (676) vert 32pt + sous-texte `[Sous Titre Champions]` (`8,5 % · ARPU 11 688`)
- KPI 2 — **At Risk** (bordure orange) : `[Nb Clients At Risk]` (1 266) orange 32pt + sous-texte `[Sous Titre At Risk]` (`15,8 % · ARPU 8 984`)
- KPI 3 — **Lost** (bordure rouge) : `[Nb Clients Lost]` (1 934) rouge 32pt + sous-texte `[Sous Titre Lost]` (`24,2 % · churn 82,2 %`)

**4. Donut — Répartition par segment RFM**
- Catégorie : `clients_rfm[Segment_RFM]`
- Valeur : `COUNTROWS(clients_rfm)`
- Couleurs via `[Color Segment RFM]`
- Trou central : 65 %

**5. Barres horizontales — Taux churn par segment RFM**
- Axe Y : `clients_rfm[Segment_RFM]`
- Axe X : taux churn par segment (Champions 0,1 % / Loyal 1,3 % / At Risk 20,5 % / Lost 82,2 %)
- Couleur : `[Color Segment RFM]`

**6. Bandeau hero — ARPU annuel à défendre**
- Carte avec bordure orange `#E94E1B` 2px + fond pâle `rgba(233,78,27,0.08)`
- Titre statique : `Top 50 At Risk - ARPU annuel a defendre`
- Sous-titre : `[Sous Titre Top 50 At Risk]` (`Tous abonnes Pro · ARPU mensuel 25 016 a 27 085 FCFA`)
- Hero number : `[ARPU Top 50 At Risk M FCFA]` (`15,6 M`) orange 32pt
- Caption : `[Caption ARPU Annuel Defendre]` (`FCFA / an si rien n'est fait`)

**7. Card des 3 leviers chiffrés — via HTML Content**
- Visuel : **HTML Content** (Daniel Marsh-Patrick)
- Champ : `[HTML Plan Action]`
- Affiche les 3 leviers numérotés avec sous-titres dynamiques + total range
- Largeur pleine, hauteur ~280 px

> 🎓 **MÉTHODE — Filtrer Top N dans un visuel Power BI**
>
> Sur le panneau **Filtres** du visuel Table :
> 1. Glisser `clients_rfm[id_client]` dans la zone **Filtres au niveau visuel**
> 2. Type de filtre : **Top N**
> 3. Afficher : **N supérieur** = `50`
> 4. Par valeur : `clients_rfm[arpu_6m]` (Somme)
> 5. Combiner avec un autre filtre `Segment_RFM = "At Risk"` pour cibler


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/powerbi/tuto/05-plan-actions.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# VII — Slicers, navigation, finitions

**Navigation horizontale en haut de chaque page** : 5 textes cliquables avec icônes « Vue Executive · Segments · Cohortes · Réclamations · Plan d'action ». Item actif : texte orange `#E94E1B` avec soulignement 2px ; items inactifs : texte gris `#6B7B95`.

> 🎓 **MÉTHODE — Navbar horizontale avec état actif**
>
> Power BI ne gère pas l'état actif natif. Astuce :
>
> 1. Sur chaque page, créer 5 boutons **Texte** (un par section), tous avec **action Navigation de page**
> 2. Sur la page courante, le bouton correspondant à la page active n'a PAS d'action de navigation et utilise un style différent (orange + soulignement)
> 3. Dupliquer la disposition des 5 boutons sur les 5 pages, en variant à chaque fois le bouton actif

**Slicer global Année** : 1 slicer déroulant `Calendrier[Annee]` (« Tout / 2024 / 2025 ») en haut à droite. Synchroniser sur toutes les pages : Format → **Synchroniser les segments → cocher toutes les pages**.

**Slicer Cohorte (page 3 uniquement)** : `clients[Cohorte_Annee]` en bouton segmenté (Format → Style → Vignette). Ne pas synchroniser avec les autres pages.

**Slicer Ville (page 2 uniquement)** : `clients[ville]` en bouton segmenté.

**Logo « IvoirCom »** : carré orange `#E94E1B` avec icône smartphone blanc en haut-gauche, suivi du nom **IvoirCom** blanc 14pt et sous-titre « Customer Churn Analytics » gris clair 11pt.

---
# VIII — Validation et livraison

## 8.1 Checklist de recette

**Modèle** :
- 6 tables sources + Calendrier + `_Mesures` (12 tables au total après suppression LocalDateTables)
- 8 relations actives + 3 inactives sur clients (date_souscription, date_resiliation, Cohorte_Mois)
- 0 LocalDateTable · `Calendrier` marquée comme table de dates · Auto Date/Time désactivé
- 8 colonnes calculées dans `clients` + 1 dans `clients_rfm`

**Mesures** : **80 dans `_Mesures`** · 8 dossiers numérotés `0.` à `6.` + `_Helpers` + `_HTML` · format défini partout (FCFA, %, mois, M FCFA, ratio).

**Pages** : 5 pages avec sous-titre dynamique · Slicer Année synchronisé · Slicer Cohorte_Annee dédié page 3 · Navbar horizontale avec état actif · Charte Dark + Orange chaleureux (`#0B1421` fond).

**Performance** : ouverture < 5 s · aucun visuel en erreur.

**Valeurs cibles à vérifier visuellement** :

| Visuel | Valeur attendue |
|---|---|
| KPI Total Clients | 8 000 |
| KPI Taux Churn | 25,4 % |
| KPI ARPU Actifs | 7 959 FCFA |
| KPI Total Réclamations | 9 791 |
| Donut Actifs/Résiliés | 5 967 / 2 033 |
| Top offre churn | Pulse 35,3 % |
| Top ville churn | Korhogo 27,8 % |
| Écart ARPU actifs/churners | +71,7 % |
| Ancienneté actifs / churners | 25,9 mois / 17,2 mois (écart +8,7) |
| Bandeau alerte signal | 1 195 clients à 2+ tickets non résolus |
| Taux churn segment risque | 93,4 % |
| Ratio signal vs base | ×3,7 |
| Tickets moyens churners / actifs | 2,53 / 0,78 (×3,2) |
| Nb Champions | 676 (8,5 %) |
| Nb At Risk | 1 266 (15,8 %) |
| Nb Lost | 1 934 (24,2 %) |
| ARPU Top 50 At Risk annuel | ~15,6 M FCFA |
| Total 3 leviers (range) | ~75-85 M FCFA |

## 8.2 Pièges fréquents

| Symptôme | Cause | Correction |
|---|---|---|
| `USERELATIONSHIP` erreur « ne peut utiliser que les 2 colonnes qui participent à la relation » | Les relations Calendrier ↔ clients pointent vers LocalDateTables | Désactiver Auto Date/Time + créer relations inactives Calendrier ↔ clients[date_souscription / date_resiliation / Cohorte_Mois] |
| `RANKX` retourne 1 partout dans une table | Pas de transition de contexte ligne → filtre | Wrapper la mesure dans `CALCULATE([Mesure])` à l'intérieur du RANKX |
| Heatmap rétention 74,6 % partout | Mesure ignorant le contexte colonne `Anciennete_Mois` | Capturer le contexte avec `VAR _M = MAX(clients[Anciennete_Mois])` puis filtrer `Anciennete_Mois >= _M` |
| Slicer Calendrier[Annee] ne filtre pas la heatmap cohortes | Relation Cohorte_Mois ↔ Calendrier inactive (chemin ambigu) | Utiliser slicer `clients[Cohorte_Annee]` à la place |
| Cohorte_Mois_Label trié alphabétiquement (jan, jul, jun...) | Tri par défaut sur la colonne texte | Outils de colonne → Trier par colonne → `Cohorte_Mois_Sort` |
| Donut Actifs/Résiliés affiche 8 030 | Doublons `_DUP` non exclus dans Power Query | Filtre Power Query : `id_client doesn't end with "_DUP"` |
| ARPU = 0 sur certains clients | Factures à montant 0 incluses | Mesure `ARPU` doit `FILTER(factures, factures[montant_fcfa] > 0)` |
| `[HTML Plan Action]` rend du texte brut | Visuel Carte au lieu de HTML Content | Installer HTML Content depuis AppSource (Daniel Marsh-Patrick) |

## 8.3 Storytelling exécutif

Pour présenter au directeur commercial IvoirCom, suis l'ordre des 5 pages :

1. **Vue Executive** : « 8 000 abonnés, taux de churn cumulatif 25,4 % (au-dessus cible 20 %). ARPU actifs 7 959 FCFA, ARPU churners 4 636 FCFA — soit +71,7 % chez les actifs. 9 791 réclamations sur la période, 47,8 % non résolues. »
2. **Segments à risque** : « Pulse 35,3 % et Étudiant 31,4 % concentrent 61 % du churn total. Toutes les villes sont autour de 25-28 % — le problème n'est pas géographique mais par offre. Cellule la plus rouge : Pulse à Korhogo. »
3. **Cohortes & Rétention** : « Rétention M+11 ~92 % sur la cohorte janvier 2024 — l'onboarding fonctionne. La dégradation est lente et régulière, donc structurelle (concurrence Orange/MTN/Moov). »
4. **Signaux Réclamations** : « 71,5 % des churners ont 2+ tickets vs 17,9 % des actifs (ratio ×3,2). Sur les 1 195 clients à 2+ tickets non résolus, 93,4 % churnent — c'est le signal le plus fort de l'analyse. »
5. **Plan d'action** : « Seulement 8,5 % de Champions (cible 15-25 %), 24,2 % de Lost. 1 266 At Risk dont 50 abonnés Pro à 25 000 FCFA mensuels = 15,6 M FCFA d'ARPU annuel exposé. »
6. **Recommandation** : 3 leviers chiffrés activables dès la semaine prochaine, sans budget ML — total ~75-85 M FCFA d'ARPU récupéré / défendu par an.

## 8.4 Annexes — Mapping mockup PPTX ↔ pages Power BI

| Slide | Background PNG | Page |
|---|---|---|
| 1 | `bg-01-vue-executive.png` | Vue Executive |
| 2 | `bg-02-segments.png` | Segments à risque |
| 3 | `bg-03-cohortes.png` | Cohortes & Rétention |
| 4 | `bg-04-reclamations.png` | Signaux Réclamations |
| 5 | `bg-05-plan-action.png` | Plan d'action |


---
<div style="background:#A8350E;padding:24px 32px;border-radius:10px;color:#FFFFFF;font-family:Georgia,serif;text-align:center;">
<div style="font-size:22px;font-weight:700;margin-bottom:6px;">Customer Churn Analytics — IvoirCom</div>
<div style="font-size:13px;color:#FFE4D6;font-family:'Segoe UI',sans-serif;"><b>DataProjectLab</b> — apprendre la data sur des cas concrets, structurés et orientés métier.</div>
</div>